In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)

try:
    from PIL import Image

    _PIL_AVAILABLE = True
except Exception:
    _PIL_AVAILABLE = False

DATA_ROOT = "/kaggle/input/plant-pathology-2020-fgvc7"

TRAIN_PATH = os.path.join(DATA_ROOT, "train.csv")
TEST_PATH = os.path.join(DATA_ROOT, "test.csv")
SAMPLE_SUB_PATH = os.path.join(DATA_ROOT, "sample_submission.csv")
IMAGES_DIR = os.path.join(DATA_ROOT, "images")



In [2]:
SUBMISSIONS_PATH = "/kaggle/input/submissions/"

submissions_all = []
if os.path.isdir(SUBMISSIONS_PATH):
    for dirname, _, filenames in os.walk(SUBMISSIONS_PATH):
        for filename in filenames:
            if filename.lower().endswith(".csv"):
                submissions_all.append(os.path.join(dirname, filename))
submissions_all.sort()
print("Found submissions:", submissions_all)




Found submissions: []


In [3]:
def model_based_submission():
    """
    Light model using several cheap image‑derived features plus 16‑bin RGB histograms
    and two extra size‑based cues (aspect ratio and bytes‑per‑pixel).  A
    RandomForestClassifier is tried first; if it also succeeds we also fit a
    HistGradientBoostingClassifier and average both predictions.  If any step
    fails we fall back to the existing hierarchical models.
    """

    import concurrent.futures

    # Updated to 62 features (12 basic + 48 histogram bins + 2 size cues)
    FEATURE_DIM = 62

    def _process_one(img_id):
        """Extract all required features for a single image."""
        path = os.path.join(IMAGES_DIR, f"{img_id}.jpg")
        if not os.path.exists(path):
            path = os.path.join(IMAGES_DIR, f"{img_id}.JPG")
        try:
            size = os.path.getsize(path)
        except Exception:
            size = 0
        num_str = "".join(filter(str.isdigit, str(img_id)))
        num_id = int(num_str) if num_str else 0

        width = height = 0
        mr = mg = mb = 0.0
        sr = sg = sb = 0.0
        gray_mean = gray_var = 0.0
        hist_r = np.zeros(16, dtype=float)
        hist_g = np.zeros(16, dtype=float)
        hist_b = np.zeros(16, dtype=float)

        if _PIL_AVAILABLE and os.path.exists(path):
            try:
                img = Image.open(path).convert("RGB")
                img_resized = img.resize((64, 64))
                img_array = np.array(img_resized)  # (64, 64, 3)

                width, height = img_resized.width, img_resized.height

                r = img_array[:, :, 0]
                g = img_array[:, :, 1]
                b = img_array[:, :, 2]

                mr = r.mean()
                mg = g.mean()
                mb = b.mean()

                sr = r.std()
                sg = g.std()
                sb = b.std()

                # 16‑bin histograms (density=True gives normalized frequencies)
                hist_r, _ = np.histogram(r, bins=16, range=(0, 255), density=True)
                hist_g, _ = np.histogram(g, bins=16, range=(0, 255), density=True)
                hist_b, _ = np.histogram(b, bins=16, range=(0, 255), density=True)

                gray = img.convert("L")
                gray_arr = np.array(gray.resize((64, 64)))
                gray_mean = gray_arr.mean()
                gray_var = gray_arr.var()
            except Exception:
                pass

        aspect_ratio = (width / height) if height > 0 else 0.0
        size_per_pixel = (size / (width * height)) if (width * height) > 0 else 0.0

        feats = np.empty(FEATURE_DIM, dtype=float)
        feats[0] = size
        feats[1] = num_id
        feats[2] = width
        feats[3] = height
        feats[4] = mr
        feats[5] = mg
        feats[6] = mb
        feats[7] = sr
        feats[8] = sg
        feats[9] = sb
        feats[10] = gray_mean
        feats[11] = gray_var
        feats[12:28] = hist_r  # 16 values
        feats[28:44] = hist_g  # 16 values
        feats[44:60] = hist_b  # 16 values
        feats[60] = aspect_ratio
        feats[61] = size_per_pixel
        return feats

    def get_features(df):
        """Parallel feature extraction preserving original order, returning a dense matrix."""
        img_ids = df["image_id"].tolist()
        n = len(img_ids)
        X = np.empty((n, FEATURE_DIM), dtype=float)

        with concurrent.futures.ProcessPoolExecutor(
            max_workers=os.cpu_count()
        ) as executor:
            for i, feats in enumerate(executor.map(_process_one, img_ids, chunksize=8)):
                X[i, :] = feats
        return X

    prob_cols = ["healthy", "multiple_diseases", "rust", "scab"]

    train_df = pd.read_csv(TRAIN_PATH)
    test_df = pd.read_csv(TEST_PATH)

    X_train = get_features(train_df)
    X_test = get_features(test_df)

    preds = np.empty((len(test_df), len(prob_cols)), dtype=float)

    for i, col in enumerate(prob_cols):
        y = train_df[col].values
        try:
            rf = RandomForestClassifier(
                n_estimators=4000,  # stronger forest
                max_depth=None,
                min_samples_leaf=1,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
            )
            rf.fit(X_train, y)
            rf_pred = rf.predict_proba(X_test)[:, 1]
        except Exception as e:
            print(f"RF failed for {col} ({e}), falling back to next model.")
            rf_pred = None

        try:
            hgb = HistGradientBoostingClassifier(
                max_iter=2500,  # more boosting iterations
                learning_rate=0.03,
                max_depth=6,
                min_samples_leaf=20,
                l2_regularization=0.1,
                random_state=42,
            )
            hgb.fit(X_train, y)
            hgb_pred = hgb.predict_proba(X_test)[:, 1]
        except Exception as e:
            print(f"HGB failed for {col} ({e}), falling back.")
            hgb_pred = None

        if rf_pred is not None and hgb_pred is not None:
            preds[:, i] = (rf_pred + hgb_pred) / 2.0
            continue

        if rf_pred is not None:
            preds[:, i] = rf_pred
            continue

        if hgb_pred is not None:
            preds[:, i] = hgb_pred
            continue

        try:
            gb = GradientBoostingClassifier(
                n_estimators=1200,
                learning_rate=0.05,
                max_depth=5,
                random_state=42,
            )
            gb.fit(X_train, y)
            preds[:, i] = gb.predict_proba(X_test)[:, 1]
            continue
        except Exception as e:
            print(f"GB also failed for {col} ({e}), using LogisticRegression.")

        lr = LogisticRegression(
            solver="liblinear",
            class_weight="balanced",
            max_iter=1000,
        )
        lr.fit(X_train, y)
        preds[:, i] = lr.predict_proba(X_test)[:, 1]

    sub_df = test_df.copy()
    sub_df[prob_cols] = preds
    sub_df.to_csv("submission.csv", index=False)
    print("Model‑based submission written to submission.csv")




In [4]:
def ensemble(submissions_all, sub_idx, weights=[]):
    """
    Weighted average of the probabilities from a list of submission files.
    If the requested indices are out of range we raise a clear error.
    """
    if not submissions_all:
        raise ValueError("No submission files available for ensembling.")
    if len(sub_idx) != len(weights):
        raise ValueError("sub_idx and weights must have the same length.")
    submission_with_weight = []
    for i in range(len(sub_idx)):
        idx = sub_idx[i]
        if idx >= len(submissions_all):
            raise IndexError(
                f"sub_idx[{i}] == {idx} is out of bounds for submissions_all of length {len(submissions_all)}"
            )
        print(f"I'm taking submission {submissions_all[idx]} with weight {weights[i]}")
        sub = pd.read_csv(submissions_all[idx])
        sub_vals = sub.loc[:, ["healthy", "multiple_diseases", "rust", "scab"]].values
        submission_with_weight.append(sub_vals * weights[i])
    submission_avg = sum(submission_with_weight)
    return submission_avg




In [5]:
def make_submission_file(submission_avg, reference_csv_path):
    """
    Write the averaged probabilities to a CSV file using the structure of a reference submission.
    """
    ref_df = pd.read_csv(reference_csv_path)
    prob_cols = ["healthy", "multiple_diseases", "rust", "scab"]
    ref_df.loc[:, prob_cols] = submission_avg
    ref_df.to_csv("submission.csv", index=False)
    print("Submission written to submission.csv")


def baseline_submission():
    """
    Simple baseline: predict each class probability as the mean label frequency
    observed in the training data.
    """
    train_df = pd.read_csv(TRAIN_PATH)
    prob_cols = ["healthy", "multiple_diseases", "rust", "scab"]
    means = train_df[prob_cols].mean().values.reshape(1, -1)
    test_df = pd.read_csv(TEST_PATH)
    baseline_preds = np.repeat(means, repeats=len(test_df), axis=0)
    sub_df = test_df.copy()
    sub_df[prob_cols] = baseline_preds
    sub_df.to_csv("submission.csv", index=False)
    print("Baseline submission written to submission.csv")




In [6]:
import numpy as np

if submissions_all:
    try:
        model_based_submission()
        model_preds = (
            pd.read_csv("submission.csv")
            .loc[:, ["healthy", "multiple_diseases", "rust", "scab"]]
            .values
        )
    except Exception as e:
        print(f"Model‑based failed ({e}), falling back to baseline.")
        baseline_submission()
        model_preds = (
            pd.read_csv("submission.csv")
            .loc[:, ["healthy", "multiple_diseases", "rust", "scab"]]
            .values
        )

    # Simple ensemble: average model prediction with the most recent prior submission if any
    try:
        # use the last available submission with equal weight
        prior_pred = (
            pd.read_csv(submissions_all[-1])
            .loc[:, ["healthy", "multiple_diseases", "rust", "scab"]]
            .values
        )
        blended = (model_preds + prior_pred) / 2.0
        make_submission_file(blended, SAMPLE_SUB_PATH)
    except Exception as e:
        print(f"Ensembling failed ({e}), using model only.")
        make_submission_file(model_preds, SAMPLE_SUB_PATH)
else:
    try:
        model_based_submission()
    except Exception as e:
        print(f"Model‑based submission failed ({e}), falling back to baseline.")
        baseline_submission()

Model‑based submission failed (Can't pickle local object 'model_based_submission.<locals>._process_one'), falling back to baseline.
Baseline submission written to submission.csv
